# just_drop vs just_drop_hal PPS

对比 **XDP ingress drop**（`examples/just_drop`）与 **HAL 应用层 drop**（`examples/just_drop_hal`）在同等 UDP 洪泛下的 DUT 处理 PPS。

数据一律经 `utils.db` **现查 Postgres**（`benchmark/.env`），不落本地结果缓存。分析方法见仓库根目录 [`xpost_hal_xdp_experiment_plan.md`](../xpost_hal_xdp_experiment_plan.md)。

---

## 复现实验流程

### 1. 准备

```bash
cd benchmark
cp .env.example .env   # 填 PG* 账号
pixi install
pixi run ensure-schema
```

固件与 agent（见 [`AGENTS.md`](../AGENTS.md)）：

```bash
# 终端 A：agent RPC
cargo xtask agent --listen 0.0.0.0:9240 --no-monitor

# 烧录 just_drop / just_drop_hal（按轮次切换）；Wi-Fi 凭据在各自 examples/*/.env
cargo xtask just_drop build --release
cargo xtask just_drop_hal build --release
```

### 2. 采集（临时 CSV，不进仓库）

在 `xpost-test` 侧（TR3000 / 已配置的 probe 环境）：

```bash
./probe just-drop-pps --path hal --out /tmp/hal_pps.csv
./probe just-drop-pps --path xdp --out /tmp/xdp_pps.csv
```

探针会旁路写出 `/tmp/hal_pps_summary.json`、`/tmp/xdp_pps_summary.json`。

> 当前探针是等间隔短窗口采样，入库后记为 `run_kind=pilot`，**不得并入正式置信区间**。规范要求的随机三点 `τ`、擦除重烧配对块、15 分钟窗口需后续采集升级。

### 3. 入库

```bash
cd benchmark
pixi run ingest-pps -- /tmp/hal_pps.csv /tmp/hal_pps_summary.json
pixi run ingest-pps -- /tmp/xdp_pps.csv /tmp/xdp_pps_summary.json
```

同一 CSV 再导入会按 `source_note` 替换先前的 pilot 行。

### 4. 分析

打开本笔记本（`pixi run lab`），向下执行：只读 `experiment_*`，画图 1 / 图 2，并标明证据边界。


## 环境与连接


In [ ]:
from utils.db import dsn_summary
from utils.experiment import (
    load_runs,
    load_counter_samples,
    load_sample_reads,
    mean_pps,
    summarize_variant,
    figure1_summary,
    figure2_noise,
    local_relative_adev,
    triad_rates,
)

print(dsn_summary())


## 从库加载 pilot 运行

`run_kind='pilot'` 表示等间隔短窗口探针数据。正式 `random` / `control` 轮到位前，下列均值与 CV **只作诊断**，不声称达到规范预注册边界。


In [ ]:
runs = load_runs(run_kind="pilot")
display(runs)

samples = load_counter_samples(runs["run_id"].tolist() if len(runs) else [])
reads = load_sample_reads(runs["run_id"].tolist() if len(runs) else [])
print(f"counter_samples={len(samples)} sample_reads={len(reads)} (triads empty → no local Allan)")


## 单轮平均 PPS 与跨轮 CV


In [ ]:
from collections import defaultdict

by_variant = defaultdict(list)
for _, row in runs.iterrows():
    sub = samples[samples["run_id"] == row["run_id"]].sort_values("time_ns")
    rp = mean_pps(
        sub["time_ns"].tolist(),
        sub["counter_value"].tolist(),
        run_id=row["run_id"],
        variant=row["variant"],
        pair_id=int(row["pair_id"]),
    )
    by_variant[row["variant"]].append(rp)

summaries = {v: summarize_variant(rs) for v, rs in by_variant.items()}
for v, s in summaries.items():
    print(f"{v}: n_valid={s['n_valid']}/{s['n_total']} mean_pps={s['mean_pps']:.1f} sd={s['sd_pps']:.1f} CV={s['cv']:.4f}")
    print("  per-run:", s["per_run_pps"])


## 图 1：单绘图区 · HAL/XDP · 三 Y 轴

左：Mean PPS；中：1 s 相对 ADEV（%）；右：跨轮 CV（%）。无三点组时 ADEV 标为 n/a，**不用**等间隔标准差顶替。


In [ ]:
hal = summaries.get("hal", {})
xdp = summaries.get("xdp", {})

# Pilot CSV has no sample_reads triads → ADEV evidence insufficient
fig1, _ = figure1_summary(
    hal_mean_pps=hal.get("mean_pps", float("nan")),
    xdp_mean_pps=xdp.get("mean_pps", float("nan")),
    hal_pps_points=hal.get("per_run_pps"),
    xdp_pps_points=xdp.get("per_run_pps"),
    hal_adev_1s=None,
    xdp_adev_1s=None,
    hal_cv=hal.get("cv"),
    xdp_cv=xdp.get("cv"),
    title="Pilot: mean PPS / ADEV(1s n/a) / CV (not formal CI)",
)
fig1


## 图 2：噪声模型诊断（正式数据不足时）

随机三点组与留出验证集尚未就绪时，此图显示「证据不足」。合成数据验收见 `pixi run test`。


In [ ]:
fig2 = figure2_noise(
    hal_adev=None,
    xdp_adev=None,
    hal_counts=None,
    xdp_counts=None,
    model_scores=None,
    title="Noise diagnostics — awaiting random triad runs",
)
fig2


## 证据边界（读库后必读）

| 结论 | 当前 pilot | 正式协议要求 |
|------|------------|--------------|
| 平均处理 PPS | 可从起终点差分估计 | 15 min、擦除重烧、共同饱和负载 |
| 1 s 局部相对 ADEV | **证据不足**（无三点组） | `τ∈{0.5…16}` 随机三点 + Δt±5% |
| 跨轮 CV | 轮数少，CI 很宽 | 12 配对块 bootstrap |
| 噪声模型 | 不拟合 pilot CSV | 前 8 拟合 / 后 4 留出 |

图与派生量只在本笔记本内显示，不写回仓库。
